In [ ]:
# install dependencies
!pip install yfinance pandas_datareader pandas_market_calendars tqdm -q


In [ ]:
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import pandas_market_calendars as mcal
from datetime import datetime
from pathlib import Path
from tqdm import tqdm

START   = "2024-03-31"
END     = "2025-03-31"
OUT_DIR = Path("macro_data")
OUT_DIR.mkdir(exist_ok=True)

# Dow Jones Industrial Average 30 constituents
TICKERS  = ["AMZN","AMGN","AXP","BA","CAT","CRM","CSCO","CVX","DIS","DOW","GS",
            "HD","HON","IBM","INTC","JNJ","JPM","KO","MCD","MMM","MRK","MSFT",
            "NKE","PG","TRV","UNH","V","VZ","WBA","WMT"]

import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import pandas_market_calendars as mcal
from datetime import datetime
from pathlib import Path
from functools import lru_cache
from tqdm import tqdm

START   = "2024-03-31"
END     = "2025-03-31"
OUT_DIR = Path("macro_csvs")
OUT_DIR.mkdir(exist_ok=True)


# Map GICS sector → Select Sector SPDR ETF
SECTOR_TO_ETF = {
    "Communication Services": "XLC",
    "Consumer Discretionary": "XLY",
    "Consumer Cyclical":      "XLY",
    "Consumer Staples":       "XLP",
    "Consumer Defensive":     "XLP",
    "Energy":                 "XLE",
    "Financial Services":     "XLF",
    "Financial":              "XLF",
    "Health Care":            "XLV",
    "Healthcare":             "XLV",
    "Industrials":            "XLI",
    "Information Technology": "XLK",
    "Technology":             "XLK",
    "Materials":              "XLB",
    "Basic Materials":        "XLB",
    "Real Estate":            "XLRE",
    "Utilities":              "XLU",
}



# Static backup for the 30 DJIA tickers – fills gaps when Yahoo fails
DJIA_STATIC = {
    "AAPL":"Technology", "AMGN":"Health Care", "AXP":"Financial Services",
    "BA":"Industrials",  "CAT":"Industrials",   "CSCO":"Technology",
    "CVX":"Energy",      "DIS":"Communication Services", "DOW":"Materials",
    "GS":"Financial Services", "HD":"Consumer Discretionary",
    "HON":"Industrials", "IBM":"Technology",    "INTC":"Technology",
    "JNJ":"Health Care", "JPM":"Financial Services", "KO":"Consumer Staples",
    "MCD":"Consumer Discretionary", "MMM":"Industrials",
    "MRK":"Health Care", "MSFT":"Technology", "NKE":"Consumer Discretionary",
    "PG":"Consumer Staples", "TRV":"Financial Services",
    "UNH":"Health Care", "V":"Financial Services", "VZ":"Communication Services",
    "WBA":"Consumer Staples", "WMT":"Consumer Staples", "RTX":"Industrials",
}

@lru_cache(maxsize=None)
def get_sector(ticker: str) -> str | None:
    """
    Robust Yahoo sector lookup:
    1. .get_info()
    2. .basic_info
    3. Search API
    4. get_growth_estimates()
    5. Static DJIA table
    """
    tk = yf.Ticker(ticker)
    # 1. get_info()
    try:
        sec = tk.get_info().get("sector")
        if sec: return sec
    except Exception:
        pass
    # 2. basic_info
    try:
        sec = tk.basic_info.get("sector")
        if sec: return sec
    except Exception:
        pass
    # 3. Search API
    try:
        q = yf.Search(ticker).quotes
        if q and "sector" in q[0]:
            return q[0]["sector"]
    except Exception:
        pass
    # 4. growth_estimates (has 'sector' column)
    try:
        ge = tk.get_growth_estimates(as_dict=False)
        if isinstance(ge, pd.DataFrame) and "sector" in ge.columns:
            sec = ge.at[0, "sector"]
            if isinstance(sec, str) and sec:
                return sec
    except Exception:
        pass
    # 5. Static fallback
    return DJIA_STATIC.get(ticker)

def map_ticker_to_etf(ticker: str) -> str | None:
    sec = get_sector(ticker)
    return SECTOR_TO_ETF.get(sec) if sec else None


# Build mapping for the current TICKERS list
TICKER_TO_ETF = {t: map_ticker_to_etf(t) for t in TICKERS}

# Unique ETF symbols actually needed
ETF_SYMBOLS = sorted({etf for etf in TICKER_TO_ETF.values() if etf})


In [ ]:

import pandas as pd, yfinance as yf, pandas_datareader.data as web
import pandas_market_calendars as mcal

# Helper: ensure tz‑naive index
def to_naive(df_or_series):
    idx = df_or_series.index
    if getattr(idx, "tz", None) is not None:
        df_or_series.index = idx.tz_localize(None)
    return df_or_series

# Helper: extract "Close" prices from a yfinance download, flat or MultiIndex
def get_close(df):
    if df.empty:
        return df
    if df.columns.nlevels == 1:                  # flat columns
        if "Close" not in df.columns:
            raise KeyError("'Close' not found in flat yfinance DataFrame")
        return df[["Close"]]
    # Multi‑Index → pick level‑1 == 'Close'
    return df.xs("Close", level=1, axis=1)

# NYSE trading‑day index (tz‑naive)
nyse = mcal.get_calendar("NYSE")
trading_days = nyse.schedule(START, END).index.normalize().tz_localize(None)

# Fed Funds Effective Rate
fed = to_naive(
    web.DataReader("DFF", "fred", START, END).rename(columns={"DFF": "FEDFUNDS"})
)

# S&P 500 close from FRED (series id = SP500)
sp500 = to_naive(
    web.DataReader("SP500", "fred", START, END)     # already column "SP500"
)

# Sector ETF Close prices we actually need
needed_etfs = sorted({etf for etf in map(map_ticker_to_etf, TICKERS) if etf})
if not needed_etfs:
    raise ValueError("No ETF symbols resolved from TICKERS.")

raw = yf.download(needed_etfs, start=START, end=END, progress=False,
                  group_by="ticker", auto_adjust=False)
sector_prices = to_naive(get_close(raw))

# If only one ETF, rename its lone column to that symbol
if isinstance(sector_prices, pd.Series):
    sector_prices = sector_prices.to_frame(name=needed_etfs[0])

#  Combine & align to trading days
macro = pd.concat([fed, sp500, sector_prices], axis=1).loc[trading_days]

print("✔ macro columns:", list(macro.columns)[:10], "…")
assert "SP500" in macro.columns


In [ ]:
rows = []

for tic in tqdm(TICKERS, desc="Tickers"):
    etf = TICKER_TO_ETF.get(tic)
    if etf is None:
        print(f"[WARN] No ETF mapping for {tic}; skipped.")
        continue

    df_tic = macro[["FEDFUNDS", "SP500", etf]].copy()
    df_tic.rename(columns={etf: "SECTOR"}, inplace=True)
    df_tic["TICKER"] = tic
    rows.append(df_tic)

combined = pd.concat(rows).reset_index().rename(columns={"index": "DATE"})
out_file = OUT_DIR / "DJIA_macro_combined.csv"
combined.to_csv(out_file, index=False)
print(f"Combined CSV saved → {out_file}  ({len(combined)} rows)")